## Recommender Systems with PySpark

#### Collaborative filtering: ALS (Alternating Least Squares)

- numBlocks (-1 imply auto-config)
- rank
- iterations
- lambda: regularization
- implicitPref
- alpha

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS

In [2]:
spark = SparkSession.builder.appName("movielens").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/06 12:26:45 WARN Utils: Your hostname, aditya-HP-Laptop-15s-eq1xxx, resolves to a loopback address: 127.0.1.1; using 192.168.88.243 instead (on interface wlo1)
26/04/06 12:26:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/06 12:26:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
df = spark.read.csv("data/ratings.csv", inferSchema=True, header=True, sep=';')

In [9]:
from pyspark.sql import functions as F
df = df.drop(F.col("timestamp"))

In [10]:
df.columns

['userId', 'movieId', 'rating']

In [12]:
df.show()

+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     1|   1193|     5|
|     1|    661|     3|
|     1|    914|     3|
|     1|   3408|     4|
|     1|   2355|     5|
|     1|   1197|     3|
|     1|   1287|     5|
|     1|   2804|     5|
|     1|    594|     4|
|     1|    919|     4|
|     1|    595|     5|
|     1|    938|     4|
|     1|   2398|     4|
|     1|   2918|     4|
|     1|   1035|     5|
|     1|   2791|     4|
|     1|   2687|     3|
|     1|   2018|     4|
|     1|   3105|     5|
|     1|   2797|     4|
+------+-------+------+
only showing top 20 rows


In [14]:
# Descriptive statistics
df.describe().show()

[Stage 7:=============================>                             (1 + 1) / 2]

+-------+-----------------+------------------+------------------+
|summary|           userId|           movieId|            rating|
+-------+-----------------+------------------+------------------+
|  count|          1000209|           1000209|           1000209|
|   mean|3024.512347919285|1865.5398981612843| 3.581564453029317|
| stddev|1728.412694899932|1096.0406894572586|1.1171018453732577|
|    min|                1|                 1|                 1|
|    max|             6040|              3952|                 5|
+-------+-----------------+------------------+------------------+



In [15]:
(train, test) = df.randomSplit([0.7, 0.3], seed=1)

In [37]:
als = ALS(maxIter=5, 
          regParam=0.01, 
          userCol="userId", 
          itemCol="movieId", 
          ratingCol = "rating", 
          coldStartStrategy="drop")

In [38]:
model = als.fit(train)

In [39]:
pred = model.transform(test)

In [40]:
pred.show()

+------+-------+------+----------+
|userId|movieId|rating|prediction|
+------+-------+------+----------+
|   148|      2|     5| 4.0129156|
|   148|     10|     4| 3.9392917|
|   148|     11|     5|  4.387577|
|   148|     44|     3| 3.6438417|
|   148|    107|     4| 3.4172451|
|   148|    153|     3| 3.4448516|
|   148|    160|     2| 3.2957258|
|   148|    161|     4| 4.1129274|
|   148|    165|     3|  4.203555|
|   148|    168|     3| 4.0490265|
|   148|    169|     3| 3.1573596|
|   148|    258|     3| 3.6437201|
|   148|    260|     5|   4.39193|
|   148|    262|     4| 3.8162107|
|   148|    293|     3| 3.6471703|
|   148|    316|     5| 3.8301697|
|   148|    327|     1| 3.1017969|
|   148|    361|     4| 4.2979393|
|   148|    364|     5|  4.311458|
|   148|    370|     4| 3.5573819|
+------+-------+------+----------+
only showing top 20 rows


In [41]:
evals = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")

In [42]:
rmse = evals.evaluate(pred)
print(f"RMSE: {rmse}")

[Stage 244:>                                                        (0 + 2) / 2]

RMSE: 0.9109970569306318


RMSE describe our error in terms of the rating column.

In [44]:
user_1 = test.filter(test['userId']==1).select(['movieId', 'userId'])

In [45]:
user_1.show()

+-------+------+
|movieId|userId|
+-------+------+
|    527|     1|
|    531|     1|
|    783|     1|
|    914|     1|
|    919|     1|
|    938|     1|
|   1022|     1|
|   1566|     1|
|   1907|     1|
|   1961|     1|
|   2018|     1|
|   2340|     1|
|   2687|     1|
|   2692|     1|
|   2762|     1|
|   3186|     1|
+-------+------+



In [46]:
rec = model.transform(user_1)

In [47]:
rec.orderBy("prediction", ascending=False).show()

+-------+------+----------+
|movieId|userId|prediction|
+-------+------+----------+
|   2018|     1| 4.8281136|
|    527|     1| 4.7212033|
|   1022|     1|  4.666644|
|    531|     1|  4.642538|
|   2687|     1| 4.6036816|
|    919|     1|  4.496803|
|   1907|     1|  4.458353|
|   1961|     1|   4.38758|
|    783|     1|  4.288147|
|    914|     1| 4.2411413|
|   2762|     1| 4.2317066|
|   3186|     1|  4.176825|
|   1566|     1| 3.8746092|
|    938|     1| 3.5325878|
|   2692|     1| 3.4003072|
|   2340|     1|  3.174646|
+-------+------+----------+

